## Part 8 – Frequency Analysis Adjustment

In this section we take a look at how far off the simple letter-frequency method actually is. 
The idea is to compare:

- the decode we get from the basic frequency mapping, and  
- the decode we get after running the Metropolis/bigram method.

By comparing the two, we can get a sense of how many cipher letters the simple method gets wrong, 
and how much adjustment the Metropolis method has to make.


In [ ]:
print(score_history[:20])


In [ ]:
# Part 8 – checking how far off simple frequency analysis actually is
# The idea here is:
#   - get a decode from the basic letter-frequency method
#   - get a decode from the Metropolis/bigram method
#   - compare the two to see how many cipher letters change their mapping
#   - also measure how different the two decodes are position-by-position
#
# This gives a rough idea of how much the simple approach needs to be fixed.

from collections import Counter

# helper: infer a mapping from (ciphertext, decoded text)
def get_mapping_from_decoded(ciphertext, decoded_text):
    m = {}
    for c in character_list:
        targets = []
        for i, ch in enumerate(ciphertext):
            if ch == c and i < len(decoded_text):
                val = decoded_text[i]
                # sometimes the decode is stored as ['A'] etc., flatten that
                if isinstance(val, list):
                    if len(val) == 1:
                        val = val[0]
                    else:
                        val = "".join(str(x) for x in val)
                targets.append(val)
        if targets:
            # majority vote, nothing fancy
            m[c] = Counter(targets).most_common(1)[0][0]
        else:
            # if the letter never appears, just map it to itself
            m[c] = c
    return m


# compare how many letters map to the same plaintext letter
def compare_maps(m1, m2):
    same, diff = 0, 0
    for c in character_list:
        if m1[c] == m2[c]:
            same += 1
        else:
            diff += 1
    return same, diff


# check how many positions match in the two decoded strings
def positional_overlap(t1, t2):
    n = min(len(t1), len(t2))
    if n == 0:
        return 0.0
    matches = sum(1 for i in range(n) if t1[i] == t2[i])
    return matches / n


# ------- run the comparison on one ciphertext -------
# (can change this to any coded_message_* we want)
ciphertext = get_coded_message(1)

# simple frequency-based decode
english_freqs = get_frequency(moby_new)
freq_map = get_letter_map_from_frequency(ciphertext, english_freqs, verbose=False)
decoded_freq = decode(ciphertext, freq_map)

# make sure we are working with plain strings, not lists
if isinstance(decoded_freq, list):
    decoded_freq = "".join(
        x[0] if isinstance(x, list) else x for x in decoded_freq
    )
    
# Metropolis / bigram decode
decoded_metro = metropolis(
    ciphertext,
    character_list,
    N=20000,
    T=1.0,
    all_bigram_freqs_dicts=bigram_stats,
    verbose=False
)[0]

# sometimes Metropolis returns a list (or list of lists) – convert to string
if isinstance(decoded_metro, list):
    decoded_metro = "".join(
        x[0] if isinstance(x, list) else x for x in decoded_metro
    )

# build inferred mappings for each decode
map_freq  = get_mapping_from_decoded(ciphertext, decoded_freq)
map_metro = get_mapping_from_decoded(ciphertext, decoded_metro)

same_letters, diff_letters = compare_maps(map_freq, map_metro)
pos_agree = positional_overlap(decoded_freq, decoded_metro)

# short summary only (no long walls of text)
print("Extension 8 – frequency vs Metropolis")
print("-------------------------------------")
print(f"Message length        : {len(ciphertext)}")
print(f"Same letter mappings  : {same_letters}")
print(f"Different mappings    : {diff_letters}")
print(f"Positional agreement  : {pos_agree*100:.2f}%")




: 

In [ ]:
# extra analysis for Q8 – keep the original Part 8 code as it is,
# and just run this cell underneath.

import matplotlib.pyplot as plt

# ------------------------
# 1) per-letter behaviour
# ------------------------

# for each cipher letter, see how often freq and Metropolis agree / disagree
def per_letter_stats(ciphertext, decoded_freq, decoded_metro):
    stats = {}
    for c in character_list:
        stats[c] = {
            "count": 0,
            "same": 0,
            "diff": 0
        }

    n = min(len(ciphertext), len(decoded_freq), len(decoded_metro))
    for i in range(n):
        c = ciphertext[i]
        f = decoded_freq[i]
        m = decoded_metro[i]
        stats[c]["count"] += 1
        if f == m:
            stats[c]["same"] += 1
        else:
            stats[c]["diff"] += 1

    for c in character_list:
        cnt = stats[c]["count"]
        if cnt > 0:
            stats[c]["error_rate"] = stats[c]["diff"] / cnt
        else:
            stats[c]["error_rate"] = None
    return stats


letter_stats = per_letter_stats(ciphertext, decoded_freq, decoded_metro)

letters = []
error_rates = []
counts = []

for c in character_list:
    if letter_stats[c]["count"] > 0 and letter_stats[c]["error_rate"] is not None:
        letters.append(c)
        error_rates.append(letter_stats[c]["error_rate"])
        counts.append(letter_stats[c]["count"])

print("Per-letter error rates (simple frequency vs Metropolis):")
for c, er, cnt in zip(letters, error_rates, counts):
    print(f"{c}: error_rate = {er:.2f}, count = {cnt}")

# bar chart: which letters are more problematic?
plt.figure()
plt.bar(range(len(letters)), error_rates)
plt.xticks(range(len(letters)), letters)
plt.xlabel("Cipher letter")
plt.ylabel("Error rate (freq vs Metropolis)")
plt.title("Letters that are more problematic for simple frequency analysis")
plt.show()


# ----------------------------------------
# 2) dependence on message length (prefixes)
# ----------------------------------------

lengths = [100, 200, 400, 800, len(ciphertext)]
lengths = [L for L in lengths if L <= len(ciphertext)]

length_results = []

for L in lengths:
    ct_sub = ciphertext[:L]

    # simple frequency-based decode on the shorter message
    english_freqs_sub = get_frequency(moby_new)
    freq_map_sub = get_letter_map_from_frequency(ct_sub, english_freqs_sub, verbose=False)
    dec_freq_sub = decode(ct_sub, freq_map_sub)

    # Metropolis/bigram decode on the shorter message
    bigram_stats_sub = all_bigram_freqs(moby_new, character_list)
    dec_metro_sub = metropolis(ct_sub, character_list, bigram_stats_sub,
                               N=20000, T=1.0, verbose=False)[0]

    agree_sub = positional_overlap(dec_freq_sub, dec_metro_sub)
    length_results.append((L, agree_sub))

print("\nLength vs positional agreement (freq vs Metropolis):")
for L, a in length_results:
    print(f"L = {L:4d}, agreement = {a*100:.2f}%")

plt.figure()
plt.plot([x[0] for x in length_results],
         [x[1] for x in length_results],
         marker="o")
plt.xlabel("Message length (characters)")
plt.ylabel("Positional agreement")
plt.title("Effect of message length on frequency-based decoding")
plt.show()


# ----------------------------------------
# 3) a rough probabilistic picture
#    P(correct | letter) vs P(letter)
# ----------------------------------------

total_len = len(ciphertext)

letter_probs = []
success_probs = []
used_letters = []

for c in character_list:
    cnt = letter_stats.get(c, {}).get("count", 0)
    er = letter_stats.get(c, {}).get("error_rate", None)
    if cnt > 0 and er is not None:
        p_c = cnt / total_len                 # empirical P(c)
        p_success = 1.0 - er                  # empirical P(correct | c)
        used_letters.append(c)
        letter_probs.append(p_c)
        success_probs.append(p_success)

print("\nLetter probability vs success probability for the simple method:")
for c, p_c, p_s in zip(used_letters, letter_probs, success_probs):
    print(f"{c}: P(c) ~ {p_c:.3f},  P(correct|c) ~ {p_s:.3f}")

plt.figure()
plt.scatter(letter_probs, success_probs)
for c, x, y in zip(used_letters, letter_probs, success_probs):
    plt.text(x, y, c)

plt.xlabel("Empirical letter probability in ciphertext")
plt.ylabel("P(correct | letter) for frequency method")
plt.title("A rough probabilistic model for the simple frequency approach")
plt.show()


In [ ]:
print(decoded_freq[:200])
print(decoded_metro[:200])
print(positional_overlap(decoded_freq, decoded_metro))


In [ ]:
s1 = score(ciphertext, all_bigram_freqs_dicts)
s2 = score(swap_of_char(ciphertext, "A", "B"), all_bigram_freqs_dicts)
print(s1, s2)
